# 🔥 Advanced PyTorch: Custom Components Deep Dive

**Assignment Part 2A — PyTorch Edition**

This notebook mirrors the TensorFlow edition, implementing all advanced constructs in PyTorch. Every major component is built from scratch: schedulers, dropout, normalization, losses, activations, metrics, layers, models, optimizers, and training loops.

### Table of Contents
1. Setup & Dataset
2. Custom Learning Rate Scheduler (OneCycle Policy)
3. Custom Dropout (MC Alpha Dropout)
4. Custom Normalization (MaxNorm Dense)
5. TensorBoard Integration
6. Custom Loss Function (Huber + Quantile)
7. Custom Activation, Initializer, Regularizer & Constraint
8. Custom Metric (Streaming Huber)
9. Custom Layers (Exponential, Dense, GaussianNoise, LayerNorm)
10. Custom Model (Residual Network)
11. Custom Optimizer (Momentum with Nesterov)
12. Custom Training Loop
13. Weights & Biases Integration

---


## 1. Setup & Dataset Preparation

In [ ]:
# ============================================================
# Install & import
# ============================================================
!pip install -q wandb torchmetrics

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.optim.lr_scheduler import _LRScheduler
from torch.utils.tensorboard import SummaryWriter
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import copy, math, os, datetime
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
# ============================================================
# Fashion MNIST
# ============================================================
transform = T.Compose([T.ToTensor()])

train_full = torchvision.datasets.FashionMNIST(
    './data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(
    './data', train=False, download=True, transform=transform)

train_set, val_set = random_split(train_full, [50000, 10000],
    generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_dataset)}")

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i, 0], cmap='gray')
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=10)
    ax.axis('off')
plt.suptitle("Fashion MNIST Samples", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Baseline CNN helper
# ============================================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def quick_train(model, epochs=15, lr=1e-3, optimizer_cls=None,
                scheduler=None, extra_fn=None):
    """Quick train/eval loop returning history dict."""
    model = model.to(device)
    if optimizer_cls:
        opt = optimizer_cls
    else:
        opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    hist = {'loss':[], 'val_loss':[], 'accuracy':[], 'val_accuracy':[], 'lr':[]}

    for epoch in range(epochs):
        model.train()
        rl, c, t = 0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            if extra_fn:
                loss = loss + extra_fn(model)
            loss.backward()
            opt.step()
            rl += loss.item()*imgs.size(0)
            c += out.argmax(1).eq(lbls).sum().item()
            t += imgs.size(0)

        hist['loss'].append(rl/t)
        hist['accuracy'].append(c/t)
        hist['lr'].append(opt.param_groups[0]['lr'])

        model.eval()
        vl, vc, vt = 0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                vl += criterion(out, lbls).item()*imgs.size(0)
                vc += out.argmax(1).eq(lbls).sum().item()
                vt += imgs.size(0)
        hist['val_loss'].append(vl/vt)
        hist['val_accuracy'].append(vc/vt)

        if scheduler:
            scheduler.step()

    return hist

print("Helpers ready.")


## 2. Custom Learning Rate Scheduler — OneCycle Policy

PyTorch's `_LRScheduler` base class lets us build custom schedules. We implement the 1cycle policy with cosine warmup and annealing.


In [ ]:
# ============================================================
# 2a — OneCycle Scheduler
# ============================================================
class OneCycleScheduler(_LRScheduler):
    """
    Leslie Smith's 1cycle policy for PyTorch.

    Phase 1 (warmup_pct of training): LR rises from max_lr/div_factor → max_lr
    Phase 2 (remaining): LR cosine-decays from max_lr → max_lr/final_div

    Args:
        optimizer: PyTorch optimizer
        max_lr: Peak learning rate
        total_steps: Total number of optimizer steps
        div_factor: Initial LR divisor (default: 25)
        final_div: Final LR divisor (default: 1e4)
        warmup_pct: Fraction of steps for warmup (default: 0.3)
    """
    def __init__(self, optimizer, max_lr, total_steps,
                 div_factor=25.0, final_div=1e4, warmup_pct=0.3):
        self.max_lr = max_lr
        self.initial_lr = max_lr / div_factor
        self.final_lr = max_lr / final_div
        self.total_steps = total_steps
        self.warmup_steps = int(total_steps * warmup_pct)
        self.decay_steps = total_steps - self.warmup_steps
        self.step_num = 0
        self.lr_history = []
        # Set initial LR
        for pg in optimizer.param_groups:
            pg['lr'] = self.initial_lr
        super().__init__(optimizer, last_epoch=-1)

    def get_lr(self):
        if self.step_num < self.warmup_steps:
            # Linear warmup
            progress = self.step_num / max(self.warmup_steps, 1)
            lr = self.initial_lr + (self.max_lr - self.initial_lr) * progress
        else:
            # Cosine annealing
            decay_progress = (self.step_num - self.warmup_steps) / max(self.decay_steps, 1)
            lr = self.final_lr + (self.max_lr - self.final_lr) * 0.5 * (
                1 + math.cos(math.pi * decay_progress))

        self.lr_history.append(lr)
        return [lr for _ in self.base_lrs]

    def step_batch(self):
        """Call this every batch instead of every epoch."""
        self.step_num += 1
        super().step()


# A/B test
print("Training with CONSTANT LR...")
base_model = SimpleCNN()
hist_const = quick_train(base_model, epochs=15, lr=1e-3)

print("Training with OneCycle...")
oc_model = SimpleCNN().to(device)
oc_opt = optim.Adam(oc_model.parameters(), lr=1e-3)
steps = len(train_loader) * 15
oc_sched = OneCycleScheduler(oc_opt, max_lr=3e-3, total_steps=steps)

# Manual loop for per-batch scheduling
criterion = nn.CrossEntropyLoss()
oc_hist = {'loss':[], 'val_loss':[], 'accuracy':[], 'val_accuracy':[]}

for epoch in range(15):
    oc_model.train()
    rl, c, t = 0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        oc_opt.zero_grad()
        out = oc_model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        oc_opt.step()
        oc_sched.step_batch()
        rl += loss.item()*imgs.size(0)
        c += out.argmax(1).eq(lbls).sum().item()
        t += imgs.size(0)
    oc_hist['loss'].append(rl/t)
    oc_hist['accuracy'].append(c/t)

    oc_model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = oc_model(imgs)
            vl += criterion(out, lbls).item()*imgs.size(0)
            vc += out.argmax(1).eq(lbls).sum().item()
            vt += imgs.size(0)
    oc_hist['val_loss'].append(vl/vt)
    oc_hist['val_accuracy'].append(vc/vt)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(hist_const['val_accuracy'], label='Constant LR')
axes[0].plot(oc_hist['val_accuracy'], label='OneCycle')
axes[0].set_title("Val Accuracy", fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(hist_const['val_loss'], label='Constant LR')
axes[1].plot(oc_hist['val_loss'], label='OneCycle')
axes[1].set_title("Val Loss", fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(oc_sched.lr_history)
axes[2].set_title("OneCycle LR Schedule", fontweight='bold')
axes[2].set_xlabel("Step"); axes[2].set_ylabel("LR")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Constant LR best val acc: {max(hist_const['val_accuracy']):.4f}")
print(f"OneCycle best val acc:    {max(oc_hist['val_accuracy']):.4f}")


## 3. Custom Dropout — MC Alpha Dropout

Alpha Dropout for SELU-based Self-Normalizing Networks, with Monte Carlo inference support.


In [ ]:
# ============================================================
# 3a — MC Alpha Dropout
# ============================================================
class MCAlphaDropout(nn.Module):
    """
    Monte Carlo Alpha Dropout for Self-Normalizing Networks.

    Replaces dropped values with the SELU saturation point
    and applies affine correction to maintain mean=0, var=1.
    Keeps dropout active during inference when mc_mode=True.

    Args:
        rate: Drop probability (default: 0.05)
    """
    ALPHA = 1.6732632423543772848
    SCALE = 1.0507009873554804934

    def __init__(self, rate=0.05):
        super().__init__()
        self.rate = rate
        self.mc_mode = False  # Toggle for MC inference

    def forward(self, x):
        if not self.training and not self.mc_mode:
            return x

        sat_val = -self.ALPHA * self.SCALE
        keep_prob = 1.0 - self.rate

        mask = (torch.rand_like(x) < keep_prob).float()
        output = x * mask + sat_val * (1.0 - mask)

        # Affine correction
        a = (keep_prob + keep_prob * (1 - keep_prob) * sat_val**2) ** (-0.5)
        b = -a * (1 - keep_prob) * sat_val

        return a * output + b

    def enable_mc(self):
        self.mc_mode = True

    def disable_mc(self):
        self.mc_mode = False


# ============================================================
# 3b — SNN with MC inference
# ============================================================
class SelfNormalizingNet(nn.Module):
    def __init__(self, dropout_rate=0.1):
        super().__init__()
        self.layers_list = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.SELU(),
            MCAlphaDropout(dropout_rate),
            nn.Linear(256, 128), nn.SELU(),
            MCAlphaDropout(dropout_rate),
            nn.Linear(128, 64), nn.SELU(),
            MCAlphaDropout(dropout_rate),
            nn.Linear(64, 10)
        )
        # LeCun init for SELU
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='linear')

    def forward(self, x):
        return self.layers_list(x)

    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, MCAlphaDropout):
                m.enable_mc()

    def disable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, MCAlphaDropout):
                m.disable_mc()


snn = SelfNormalizingNet(0.1)
print("Training Self-Normalizing Network...")
snn_hist = quick_train(snn, epochs=20)

# MC inference
snn.eval()
snn.enable_mc_dropout()

test_imgs = torch.cat([imgs for imgs, _ in test_loader])[:500].to(device)
test_lbls = torch.cat([lbls for _, lbls in test_loader])[:500].numpy()

mc_preds = torch.stack([
    F.softmax(snn(test_imgs), dim=1)
    for _ in range(30)
]).cpu().numpy()

mean_preds = mc_preds.mean(axis=0)
entropy = -np.sum(mean_preds * np.log(mean_preds + 1e-10), axis=1)
mc_cls = mean_preds.argmax(axis=1)

snn.disable_mc_dropout()
std_cls = F.softmax(snn(test_imgs), dim=1).argmax(1).cpu().numpy()

print(f"Standard acc: {(std_cls == test_lbls).mean():.4f}")
print(f"MC Dropout acc: {(mc_cls == test_lbls).mean():.4f}")

correct = mc_cls == test_lbls
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(entropy[correct], bins=25, alpha=0.7, color='green', label='Correct')
ax.hist(entropy[~correct], bins=25, alpha=0.7, color='red', label='Wrong')
ax.set_title("MC Alpha Dropout — Uncertainty", fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Custom Normalization — MaxNorm Dense

Constrains each neuron's incoming weight vector to have at most `max_norm` L2 norm.


In [ ]:
# ============================================================
# 4 — MaxNorm Dense Layer
# ============================================================
class MaxNormLinear(nn.Module):
    """
    Linear layer with max-norm weight clipping.
    After each forward pass during training, weight vectors
    are rescaled so their L2 norm ≤ max_norm.

    Args:
        in_features: Input dimension
        out_features: Output dimension
        max_norm: Maximum L2 norm per neuron (default: 1.0)
    """
    def __init__(self, in_features, out_features, max_norm=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.max_norm = max_norm

    def forward(self, x):
        if self.training:
            with torch.no_grad():
                # Clip each column (neuron) of the weight matrix
                norms = self.linear.weight.norm(dim=1, keepdim=True)
                scale = torch.clamp(norms / self.max_norm, min=1.0)
                self.linear.weight.div_(scale)
        return self.linear(x)


# Test
maxnorm_net = nn.Sequential(
    nn.Flatten(),
    MaxNormLinear(784, 256, max_norm=1.0), nn.ReLU(),
    MaxNormLinear(256, 128, max_norm=1.0), nn.ReLU(),
    MaxNormLinear(128, 10, max_norm=2.0)
)

print("Training MaxNorm network...")
mn_hist = quick_train(maxnorm_net, epochs=15)
print(f"Best val acc: {max(mn_hist['val_accuracy']):.4f}")

# Verify norms
for i, m in enumerate(maxnorm_net):
    if isinstance(m, MaxNormLinear):
        norms = m.linear.weight.norm(dim=1)
        print(f"  Layer {i}: max norm = {norms.max():.4f} (limit: {m.max_norm})")


## 5. TensorBoard Integration

In [ ]:
# ============================================================
# 5 — TensorBoard with detailed logging
# ============================================================
log_dir = f"runs/pt_advanced_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir)

tb_model = SimpleCNN().to(device)
optimizer = optim.Adam(tb_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Log model graph
writer.add_graph(tb_model, torch.randn(1, 1, 28, 28).to(device))

print(f"Training with TensorBoard → {log_dir}")

for epoch in range(15):
    tb_model.train()
    rl, c, t = 0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = tb_model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        rl += loss.item()*imgs.size(0)
        c += out.argmax(1).eq(lbls).sum().item()
        t += imgs.size(0)

    train_loss, train_acc = rl/t, c/t

    tb_model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = tb_model(imgs)
            vl += criterion(out, lbls).item()*imgs.size(0)
            vc += out.argmax(1).eq(lbls).sum().item()
            vt += imgs.size(0)

    val_loss, val_acc = vl/vt, vc/vt

    # Log scalars
    writer.add_scalars('Loss', {'train': train_loss, 'val': val_loss}, epoch)
    writer.add_scalars('Accuracy', {'train': train_acc, 'val': val_acc}, epoch)
    writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)

    # Log weight histograms
    if epoch % 3 == 0:
        for name, param in tb_model.named_parameters():
            writer.add_histogram(f'weights/{name}', param.data, epoch)
            if param.grad is not None:
                writer.add_histogram(f'grads/{name}', param.grad, epoch)

writer.close()
print(f"Val acc: {val_acc:.4f}")
print("\nView: %load_ext tensorboard")
print(f"       %tensorboard --logdir {log_dir}")


## 6. Custom Loss Functions

In [ ]:
# ============================================================
# 6a — Huber Loss
# ============================================================
class HuberLoss(nn.Module):
    """
    Huber Loss: quadratic for small errors, linear for large.
    More robust to outliers than MSE.

    |error| ≤ delta: 0.5 * error²
    |error| > delta: delta * |error| - 0.5 * delta²
    """
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta

    def forward(self, y_pred, y_true):
        error = y_true - y_pred
        abs_error = torch.abs(error)
        quadratic = 0.5 * error ** 2
        linear = self.delta * abs_error - 0.5 * self.delta ** 2
        return torch.where(abs_error <= self.delta, quadratic, linear).mean()


class QuantileLoss(nn.Module):
    """Asymmetric loss for quantile regression. tau=0.5 → median."""
    def __init__(self, tau=0.5):
        super().__init__()
        self.tau = tau

    def forward(self, y_pred, y_true):
        error = y_true - y_pred
        return torch.max(self.tau * error, (self.tau - 1) * error).mean()


# Demo with outliers
np.random.seed(42)
X_reg = np.random.rand(2000, 1).astype(np.float32) * 10
y_reg = 2.5 * X_reg.flatten() + 5 + np.random.randn(2000).astype(np.float32) * 2
outliers = np.random.choice(2000, 50, replace=False)
y_reg[outliers] += np.random.randn(50).astype(np.float32) * 30

X_t = torch.tensor(X_reg)
y_t = torch.tensor(y_reg)

def train_reg(loss_fn, epochs=80):
    model = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        opt.zero_grad()
        loss_fn(model(X_t).squeeze(), y_t).backward()
        opt.step()
    return model

mse_model = train_reg(nn.MSELoss())
huber_model = train_reg(HuberLoss(delta=2.0))

X_plot = torch.linspace(0, 10, 100).unsqueeze(1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_reg, y_reg, alpha=0.2, s=10, label='Data')
ax.plot(X_plot.numpy(), mse_model(X_plot).detach().numpy(), 'r-', lw=2, label='MSE')
ax.plot(X_plot.numpy(), huber_model(X_plot).detach().numpy(), 'g-', lw=2, label='Huber')
ax.plot(X_plot.numpy(), 2.5*X_plot.numpy()+5, 'k--', lw=1, label='True')
ax.set_title("MSE vs Huber Loss (with Outliers)", fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Custom Activation, Initializer, Regularizer & Constraint

In [ ]:
# ============================================================
# 7a — Parametric Swish Activation
# ============================================================
class ParametricSwish(nn.Module):
    """
    x * sigmoid(beta * x) where beta is learnable per-channel.
    When beta=1, this is standard Swish (SiLU).
    """
    def __init__(self, num_features):
        super().__init__()
        self.beta = nn.Parameter(torch.ones(num_features))

    def forward(self, x):
        return x * torch.sigmoid(self.beta * x)


# 7b — Custom Initializer
def variance_scaling_init_(tensor, scale_factor=2.0):
    """He-like init with configurable scale factor."""
    fan_in = tensor.shape[1] if tensor.dim() >= 2 else tensor.shape[0]
    fan_out = tensor.shape[0] if tensor.dim() >= 2 else 1
    std = math.sqrt(scale_factor / ((fan_in + fan_out) / 2.0))
    with torch.no_grad():
        tensor.normal_(0, std)
    return tensor


# 7c — Spectral Regularizer (as a function added to loss)
class SpectralRegularizer:
    """
    Penalizes the largest singular value of weight matrices.
    Bounds the Lipschitz constant of the layer.
    """
    def __init__(self, model, strength=0.01):
        self.model = model
        self.strength = strength

    def __call__(self):
        penalty = 0.0
        for p in self.model.parameters():
            if p.dim() >= 2:
                # SVD for largest singular value
                u, s, v = torch.svd(p.view(p.size(0), -1))
                penalty += s[0]  # largest singular value
        return self.strength * penalty


# 7d — Non-negative weight constraint (applied after optimizer step)
def apply_nonneg_constraint(model):
    """Clamp all weights to be >= 0 after each update."""
    with torch.no_grad():
        for p in model.parameters():
            if p.dim() >= 2:
                p.clamp_(min=0.0)


# 7e — Combined model
class CustomComponentsNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.act1 = ParametricSwish(256)
        self.fc2 = nn.Linear(256, 128)
        self.act2 = ParametricSwish(128)
        self.fc3 = nn.Linear(128, 10)

        # Apply custom init
        variance_scaling_init_(self.fc1.weight, scale_factor=2.0)
        variance_scaling_init_(self.fc2.weight, scale_factor=2.0)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        return self.fc3(x)


custom_net = CustomComponentsNet()
spec_reg = SpectralRegularizer(custom_net, 0.01)

print("Training with custom activation + init + spectral reg...")
hist_custom = quick_train(custom_net, epochs=15, extra_fn=lambda m: spec_reg())
print(f"Best val acc: {max(hist_custom['val_accuracy']):.4f}")
print("\n✓ ParametricSwish, variance_scaling_init, SpectralRegularizer, NonNeg constraint")


## 8. Custom Metric — Top-K Accuracy

In [ ]:
# ============================================================
# 8 — Streaming Top-K Accuracy Metric
# ============================================================
class TopKAccuracy:
    """
    Streaming top-k accuracy metric for PyTorch.
    Tracks correct/total across batches, computes at epoch end.
    """
    def __init__(self, k=3):
        self.k = k
        self.correct = 0
        self.total = 0

    def update(self, y_pred, y_true):
        top_k = y_pred.topk(self.k, dim=1).indices
        matches = (top_k == y_true.unsqueeze(1)).any(dim=1)
        self.correct += matches.sum().item()
        self.total += y_true.size(0)

    def compute(self):
        return self.correct / max(self.total, 1)

    def reset(self):
        self.correct = 0
        self.total = 0


class HuberMetric:
    """Streaming Huber metric."""
    def __init__(self, delta=1.0):
        self.delta = delta
        self.total = 0.0
        self.count = 0

    def update(self, y_pred, y_true):
        error = (y_true.float() - y_pred.float()).abs()
        quad = torch.clamp(error, max=self.delta)
        lin = error - quad
        huber = 0.5 * quad**2 + self.delta * lin
        self.total += huber.sum().item()
        self.count += y_true.numel()

    def compute(self):
        return self.total / max(self.count, 1)

    def reset(self):
        self.total = 0.0
        self.count = 0

# Demo
topk = TopKAccuracy(k=3)
model_for_metric = SimpleCNN().to(device)
_ = quick_train(model_for_metric, epochs=10)

model_for_metric.eval()
topk.reset()
acc1_total, acc1_count = 0, 0

with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        out = model_for_metric(imgs)
        topk.update(out, lbls)
        acc1_total += out.argmax(1).eq(lbls).sum().item()
        acc1_count += lbls.size(0)

print(f"Top-1 Accuracy: {acc1_total/acc1_count:.4f}")
print(f"Top-3 Accuracy: {topk.compute():.4f}")


## 9. Custom Layers

In [ ]:
# ============================================================
# 9a — Exponential Layer
# ============================================================
class ExponentialLayer(nn.Module):
    """Element-wise exponential: output = exp(input). No parameters."""
    def forward(self, x):
        return torch.exp(x)

print(f"ExponentialLayer([0,1,2]) = {ExponentialLayer()(torch.tensor([0.,1.,2.]))}")


In [ ]:
# ============================================================
# 9b — Custom Dense (Linear) from scratch
# ============================================================
class MyLinear(nn.Module):
    """
    Linear layer from scratch: y = xW^T + b.
    Demonstrates Parameter, reset_parameters, and forward.
    """
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.in_features
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        out = x @ self.weight.t()
        if self.bias is not None:
            out = out + self.bias
        return out

test_out = MyLinear(8, 4)(torch.randn(2, 8))
print(f"MyLinear(8→4): output shape = {test_out.shape}")


In [ ]:
# ============================================================
# 9c — Gaussian Noise Layer
# ============================================================
class AddGaussianNoise(nn.Module):
    """Adds Gaussian noise during training only."""
    def __init__(self, stddev=0.1):
        super().__init__()
        self.stddev = stddev

    def forward(self, x):
        if self.training:
            return x + torch.randn_like(x) * self.stddev
        return x


In [ ]:
# ============================================================
# 9d — Custom Layer Normalization
# ============================================================
class MyLayerNorm(nn.Module):
    """
    Layer Normalization from scratch.
    Normalizes across the last dimension per-sample (not per-batch).
    """
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta


# Combined model with all custom layers
all_custom = nn.Sequential(
    nn.Flatten(),
    AddGaussianNoise(0.1),
    MyLinear(784, 256), nn.ReLU(),
    MyLayerNorm(256),
    MyLinear(256, 128), nn.ReLU(),
    MyLayerNorm(128),
    MyLinear(128, 10)
)

print("Training with all custom layers...")
hist_cl = quick_train(all_custom, epochs=15)
print(f"Best val acc: {max(hist_cl['val_accuracy']):.4f}")


## 10. Custom Model — Residual Network

In [ ]:
# ============================================================
# 10 — Residual Block + Classifier
# ============================================================
class ResidualBlock(nn.Module):
    """
    Residual block: output = F(x) + x
    Projection shortcut when dimensions change.
    """
    def __init__(self, in_features, out_features, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.BatchNorm1d(out_features),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_features, out_features),
            nn.BatchNorm1d(out_features),
        )
        self.shortcut = (nn.Linear(in_features, out_features, bias=False)
                        if in_features != out_features else nn.Identity())
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))


class ResidualClassifier(nn.Module):
    """
    Deep residual classifier using Model subclassing pattern.
    Input → Flatten → Dense → [ResidualBlock x N] → Output
    """
    def __init__(self, num_classes=10, num_blocks=4, block_units=128, dropout=0.2):
        super().__init__()
        self.flatten = nn.Flatten()
        self.input_layer = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU())

        blocks = [ResidualBlock(256, block_units, dropout)]
        for _ in range(num_blocks - 1):
            blocks.append(ResidualBlock(block_units, block_units, dropout))
        self.res_blocks = nn.Sequential(*blocks)
        self.output_layer = nn.Linear(block_units, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.input_layer(x)
        x = self.res_blocks(x)
        return self.output_layer(x)


res_model = ResidualClassifier(num_blocks=4, block_units=128)
print("Training Residual Classifier (4 blocks)...")
hist_res = quick_train(res_model, epochs=20)
print(f"Best val acc: {max(hist_res['val_accuracy']):.4f}")
print(f"Parameters: {sum(p.numel() for p in res_model.parameters()):,}")


## 11. Custom Optimizer — Momentum with Nesterov

In [ ]:
# ============================================================
# 11 — Custom SGD + Nesterov Momentum
# ============================================================
class MyMomentumOptimizer(optim.Optimizer):
    """
    SGD with Momentum from scratch, with optional Nesterov look-ahead.

    Standard:  v = β·v + grad,  θ = θ - lr·v
    Nesterov:  v = β·v + grad,  θ = θ - lr·(β·v + grad)

    Args:
        params: Model parameters
        lr: Learning rate (default: 0.01)
        momentum: Momentum coefficient (default: 0.9)
        nesterov: Use Nesterov acceleration (default: True)
    """
    def __init__(self, params, lr=0.01, momentum=0.9, nesterov=True):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            momentum = group['momentum']
            nesterov = group['nesterov']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad

                # Get or initialize velocity buffer
                state = self.state[p]
                if 'velocity' not in state:
                    state['velocity'] = torch.zeros_like(p.data)

                v = state['velocity']

                # Update velocity
                v.mul_(momentum).add_(grad)

                if nesterov:
                    # Nesterov: look-ahead update
                    p.add_(momentum * v + grad, alpha=-lr)
                else:
                    # Standard momentum
                    p.add_(v, alpha=-lr)

        return loss


# A/B: Adam vs Custom Momentum
print("Training with Custom Nesterov Momentum...")
mom_model = SimpleCNN()
mom_opt = MyMomentumOptimizer(mom_model.parameters(), lr=0.01, momentum=0.9, nesterov=True)
hist_mom = quick_train(mom_model, epochs=15, optimizer_cls=mom_opt)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_const['val_accuracy'], label='Adam (baseline)')
ax.plot(hist_mom['val_accuracy'], label='Custom Nesterov')
ax.set_title("Optimizer Comparison", fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Adam best val acc:     {max(hist_const['val_accuracy']):.4f}")
print(f"Nesterov best val acc: {max(hist_mom['val_accuracy']):.4f}")


## 12. Custom Training Loop

Full manual training loop with gradient clipping, metric tracking, and progress reporting.


In [ ]:
# ============================================================
# 12 — Custom training loop
# ============================================================
loop_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 300), nn.ReLU(),
    nn.Linear(300, 100), nn.ReLU(),
    nn.Linear(100, 10)
).to(device)

optimizer = optim.Adam(loop_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 15
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'grad_norms':[]}
best_val_loss = float('inf')
best_state = None

print("Custom Training Loop — Fashion MNIST")
print("=" * 65)

for epoch in range(EPOCHS):
    # ---- Training ----
    loop_model.train()
    running_loss, correct, total = 0.0, 0, 0
    batch_grad_norms = []

    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        # Forward
        outputs = loop_model(imgs)
        loss = criterion(outputs, lbls)

        # Backward with gradient clipping
        optimizer.zero_grad()
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(loop_model.parameters(), max_norm=5.0)
        batch_grad_norms.append(grad_norm.item())
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        correct += outputs.argmax(1).eq(lbls).sum().item()
        total += imgs.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    avg_grad = np.mean(batch_grad_norms)

    # ---- Validation ----
    loop_model.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = loop_model(imgs)
            vl += criterion(out, lbls).item() * imgs.size(0)
            vc += out.argmax(1).eq(lbls).sum().item()
            vt += imgs.size(0)

    val_loss = vl / vt
    val_acc = vc / vt

    scheduler.step(val_loss)

    # Track best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(loop_model.state_dict())

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['grad_norms'].append(avg_grad)

    print(f"  Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
          f"Acc: {train_acc:.4f}/{val_acc:.4f} | "
          f"Grad: {avg_grad:.3f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")

# Restore best and test
loop_model.load_state_dict(best_state)
loop_model.eval()
tc, tt = 0, 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        tc += loop_model(imgs).argmax(1).eq(lbls).sum().item()
        tt += lbls.size(0)
print(f"\nTest Accuracy: {tc/tt:.4f}")


In [ ]:
# ============================================================
# 12b — Visualize
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title("Loss", fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title("Accuracy", fontweight='bold'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['grad_norms'], color='orange')
axes[2].set_title("Gradient Norms", fontweight='bold'); axes[2].grid(True, alpha=0.3)

plt.suptitle("Custom Training Loop", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 13. Weights & Biases Integration

In [ ]:
# ============================================================
# 13 — W&B Training
# ============================================================
import wandb

wandb.init(
    project="advanced-pytorch-constructs",
    name="fashion-mnist-pytorch",
    config={
        "architecture": "CNN",
        "dataset": "Fashion-MNIST",
        "epochs": 15,
        "batch_size": 128,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "dropout": 0.3,
    },
    mode="offline"
)

config = wandb.config
wandb_model = SimpleCNN().to(device)
optimizer = optim.Adam(wandb_model.parameters(), lr=config.learning_rate)
criterion = nn.CrossEntropyLoss()

print(f"W&B run: {wandb.run.name}")
print("Training with W&B logging...")

for epoch in range(config.epochs):
    wandb_model.train()
    rl, c, t = 0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = wandb_model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        rl += loss.item()*imgs.size(0)
        c += out.argmax(1).eq(lbls).sum().item()
        t += imgs.size(0)

    wandb_model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = wandb_model(imgs)
            vl += criterion(out, lbls).item()*imgs.size(0)
            vc += out.argmax(1).eq(lbls).sum().item()
            vt += imgs.size(0)

    metrics = {
        "epoch": epoch,
        "train_loss": rl/t, "train_acc": c/t,
        "val_loss": vl/vt, "val_acc": vc/vt,
    }
    wandb.log(metrics)

    # Log predictions table every 5 epochs
    if epoch % 5 == 0:
        sample_imgs, sample_lbls = next(iter(val_loader))
        sample_imgs = sample_imgs[:16].to(device)
        preds = wandb_model(sample_imgs).argmax(1).cpu()

        table = wandb.Table(columns=["Image", "Predicted", "True", "Correct"])
        for i in range(16):
            img = wandb.Image(sample_imgs[i].cpu())
            table.add_data(img, CLASS_NAMES[preds[i]], CLASS_NAMES[sample_lbls[i]],
                          preds[i].item() == sample_lbls[i].item())
        wandb.log({f"predictions_epoch_{epoch}": table})

# Test
wandb_model.eval()
tc, tt = 0, 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        tc += wandb_model(imgs).argmax(1).eq(lbls).sum().item()
        tt += lbls.size(0)

wandb.log({"test_accuracy": tc/tt})
wandb.log({"total_params": sum(p.numel() for p in wandb_model.parameters())})
wandb.finish()

print(f"\nTest Accuracy: {tc/tt:.4f}")
print("W&B run finished. Use mode='online' + wandb.login() for cloud dashboard.")


## Summary

| # | Component | PyTorch Implementation | Key Insight |
|---|-----------|----------------------|-------------|
| 1 | **LR Scheduler** | `OneCycleScheduler(_LRScheduler)` | Cosine warmup+decay |
| 2 | **Custom Dropout** | `MCAlphaDropout(nn.Module)` | SELU-compatible MC inference |
| 3 | **Custom Norm** | `MaxNormLinear(nn.Module)` | Per-neuron weight clipping |
| 4 | **TensorBoard** | `SummaryWriter` | Scalars, histograms, graphs |
| 5 | **Custom Loss** | `HuberLoss(nn.Module)` | Outlier-robust regression |
| 6 | **Custom Activation** | `ParametricSwish(nn.Module)` | Learnable beta per channel |
| 6 | **Custom Initializer** | `variance_scaling_init_()` | In-place fan-based init |
| 6 | **Custom Regularizer** | `SpectralRegularizer` | σ_max penalty on weights |
| 6 | **Custom Constraint** | `apply_nonneg_constraint()` | Post-step weight clamp |
| 7 | **Custom Metric** | `TopKAccuracy` class | Streaming accumulator |
| 8 | **Custom Layers** | `MyLinear`, `AddGaussianNoise`, `MyLayerNorm` | Full Module API |
| 9 | **Custom Model** | `ResidualClassifier(nn.Module)` | Skip connections |
| 10 | **Custom Optimizer** | `MyMomentumOptimizer(Optimizer)` | Nesterov from scratch |
| 11 | **Training Loop** | Manual forward/backward/step | Full control |
| 12 | **W&B** | `wandb.log()`, `wandb.Table` | Experiment tracking |

---
*Notebook generated for Assignment Part 2A — PyTorch Edition*
